# Applied Machine Learning — Recipe Rating Prediction

This notebook builds a machine learning framework to predict recipe ratings using ingredients, nutrition information, recipe complexity, popularity, and recipe metadata.

The features are organised into several main groups:

- **Ingredient classes**: broader ingredient groups such as dairy, meat/poultry, seafood, vegetables, grains/starches, herbs/spices, sauces/condiments, and sweeteners.
- **Individual ingredient flags**: common ingredients represented as binary indicators showing whether each ingredient appears in a recipe.
- **Recipe complexity**: features such as ingredient count, number of instruction steps, preparation/cooking time, text length, and ratios such as ingredients per step.
- **Nutrition**: calories, macronutrients, log-transformed nutrition values, and macro-per-calorie ratios.
- **Metadata/context**: publication timing and recipe source/category information.
- **Popularity signal**: rating count, which can help explain historical ratings but may be less useful for predicting brand-new recipes.

The main modelling task is regression, where the goal is to predict the actual rating value a recipe is likely to receive. The champion model is **Gradient Boosting Regressor**, selected because it achieved the strongest test performance in the model comparison. Challenger models are also included to compare performance against alternative approaches, including Random Forest Regressor and simpler baseline models.

## How to use

Place these files in the same folder as this notebook:

1. `allrecipes_all.csv`
2. `recipes_ingredients_long.csv`
3. this notebook

Then run the notebook from top to bottom.

Important switches in `RecipeModelConfig`:

- `use_ingredient_group_features=True` adds interpretable ingredient-class features.
- `use_individual_ingredient_flags=True` keeps frequent individual ingredients as detailed signals.
- `use_rating_count_as_feature=True` improves historical prediction but should be set to `False` for cold-start prediction.
- `quick_run=True` can be used for a faster classroom/demo run.


In [88]:
%pip install plotly statsmodels

Note: you may need to restart the kernel to use updated packages.


## 0 — Imports and configuration

In [89]:
import re, ast, warnings
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.feature_selection import f_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore")
px.defaults.template = "plotly_white"


@dataclass
class RecipeModelConfig:
    # Files
    allrecipes_file: str = "allrecipes_all.csv"
    ingredients_file: str = "recipes_ingredients_long.csv"
    excluded_raw_columns: Tuple[str, ...] = ("author",)

    # Target and split
    target: str = "rating_value"
    filter_low_ratings: bool = True
    min_target_value: float = 4.0
    max_target_value: float = 5.0
    min_rating_count: int = 3
    test_size: float = 0.20
    random_state: int = 42
    use_stratified_split: bool = True

    # Component settings
    top_n_ingredients: int = 60
    top_n_source_categories: int = 15
    use_individual_ingredient_flags: bool = True
    use_ingredient_group_features: bool = True
    use_rating_count_as_feature: bool = True
    use_sample_weights: bool = True

    # Heuristic ingredient-class dictionary.
    # These are intentionally simple and transparent so they can be explained in the report.
    ingredient_group_patterns: Dict[str, Tuple[str, ...]] = field(default_factory=lambda: {
        "meat_poultry": (
            r"chicken", r"turkey", r"beef", r"pork", r"bacon", r"ham", r"sausage", r"lamb", r"veal",
            r"prosciutto", r"pepperoni", r"chorizo", r"ground meat", r"steak"
        ),
        "seafood": (
            r"salmon", r"tuna", r"shrimp", r"fish", r"cod", r"crab", r"lobster", r"clam", r"oyster",
            r"anchovy", r"sardine", r"trout", r"tilapia", r"scallop"
        ),
        "dairy": (
            r"milk", r"cream", r"cheese", r"butter", r"yogurt", r"sour cream", r"half and half",
            r"half-and-half", r"mozzarella", r"parmesan", r"cheddar", r"feta", r"ricotta"
        ),
        "eggs": (r"\begg\b", r"\beggs\b"),
        "vegetables": (
            r"onion", r"garlic", r"tomato", r"pepper", r"potato", r"carrot", r"celery", r"spinach",
            r"broccoli", r"mushroom", r"zucchini", r"lettuce", r"corn", r"cabbage", r"cucumber", r"squash"
        ),
        "fruit": (
            r"apple", r"banana", r"berry", r"berries", r"orange", r"lemon", r"lime", r"pineapple", r"mango",
            r"peach", r"pear", r"raisin", r"cranberry", r"avocado", r"coconut"
        ),
        "grains_starches": (
            r"flour", r"rice", r"pasta", r"noodle", r"bread", r"tortilla", r"oat", r"quinoa", r"cornmeal",
            r"cereal", r"breadcrumb", r"cracker", r"dough", r"pizza crust"
        ),
        "legumes_soy": (r"bean", r"chickpea", r"lentil", r"pea", r"tofu", r"soy"),
        "nuts_seeds": (
            r"almond", r"walnut", r"pecan", r"peanut", r"cashew", r"pistachio", r"sesame", r"chia", r"flax",
            r"sunflower seed", r"pumpkin seed"
        ),
        "herbs_spices_seasonings": (
            r"basil", r"oregano", r"parsley", r"cilantro", r"thyme", r"rosemary", r"cumin", r"paprika",
            r"pepper", r"cinnamon", r"nutmeg", r"ginger", r"turmeric", r"coriander", r"dill", r"seasoning", r"salt"
        ),
        "oils_fats": (r"oil", r"olive oil", r"vegetable oil", r"canola", r"shortening", r"lard", r"margarine", r"mayonnaise"),
        "sweeteners_chocolate": (r"sugar", r"honey", r"syrup", r"molasses", r"jam", r"jelly", r"chocolate", r"cocoa", r"caramel"),
        "sauces_condiments": (
            r"sauce", r"soy sauce", r"ketchup", r"mustard", r"vinegar", r"dressing", r"salsa", r"hot sauce",
            r"worcestershire", r"barbecue", r"bbq", r"miso", r"pesto"
        ),
        "baking_additives": (r"baking powder", r"baking soda", r"yeast", r"vanilla", r"cornstarch", r"gelatin", r"pectin"),
        "beverages_alcohol_broth": (r"wine", r"beer", r"liqueur", r"rum", r"vodka", r"coffee", r"tea", r"juice", r"broth", r"stock")
    })

    # Data controls
    control_mode: str = "filter"       # "report", "filter", or "stop"
    use_missing_control: bool = True
    use_low_variance_control: bool = True
    use_outlier_control: bool = True
    use_correlation_control: bool = True
    use_vif_control: bool = True
    max_missing_rate: float = 0.60
    min_unique_values: int = 2
    iqr_k: float = 1.5
    outlier_action: str = "winsorize"  # "none", "winsorize", or "remove_rows"
    corr_threshold: float = 0.90
    vif_threshold: float = 10.0

    # Modeling
    run_cross_validation: bool = True
    cv_folds: int = 3
    n_jobs: int = 1
    champion_model_name: str = "Champion — Gradient Boosting Regressor"

    # Feature significance
    run_feature_significance: bool = True
    ols_max_features: int = 30
    significance_level: float = 0.05

    # Optional speed switch for demos
    quick_run: bool = False
    quick_run_rows: Optional[int] = 5000


CONFIG = RecipeModelConfig()


## 1 — Shared utility functions and model wrappers

In [90]:
def extract_number(x):
    """Extract the first numeric value from strings such as '366 calories', '12g', or '1,200 mg'."""
    if pd.isna(x):
        return np.nan
    match = re.search(r"[-+]?\d*\.?\d+", str(x).replace(",", ""))
    return float(match.group()) if match else np.nan


def parse_minutes(x):
    """Parse ISO-8601 duration or simple text duration into minutes."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    iso = re.fullmatch(r"P(?:(\d+)D)?(?:T(?:(\d+)H)?(?:(\d+)M)?)?", s.upper())
    if iso:
        return 1440 * float(iso.group(1) or 0) + 60 * float(iso.group(2) or 0) + float(iso.group(3) or 0)
    hours = re.search(r"(\d+(?:\.\d+)?)\s*(?:hour|hours|hr|hrs|h)\b", s)
    mins = re.search(r"(\d+(?:\.\d+)?)\s*(?:minute|minutes|min|mins|m)\b", s)
    if hours or mins:
        total = 0
        if hours:
            total += 60 * float(hours.group(1))
        if mins:
            total += float(mins.group(1))
        return total
    return extract_number(s)


def safe_literal_list(x):
    """Convert stringified list cells to Python lists. Return [] when not usable."""
    if pd.isna(x):
        return []
    try:
        y = ast.literal_eval(str(x))
        if isinstance(y, list):
            return [str(i).strip().lower() for i in y if str(i).strip()]
        return [str(y).strip().lower()]
    except Exception:
        return [str(x).strip().lower()] if str(x).strip() else []


def count_items(x):
    """Count items in list-like fields or instruction text."""
    if pd.isna(x):
        return np.nan
    try:
        y = ast.literal_eval(str(x))
        if isinstance(y, list):
            return len(y)
    except Exception:
        pass
    parts = re.split(r"[.;\n]+", str(x))
    return len([p for p in parts if p.strip()])


def safe_divide(a, b):
    """Vectorized division that returns NaN for invalid denominators."""
    return np.where((pd.notna(b)) & (b != 0), a / b, np.nan)


def make_safe_column_name(value, prefix=""):
    """Create readable, model-safe column names."""
    clean = re.sub(r"[^a-zA-Z0-9]+", "_", str(value).lower()).strip("_")[:50]
    clean = clean or "missing"
    return f"{prefix}{clean}"


def make_unique(names):
    """Keep column names unique after sanitizing text labels."""
    seen = {}
    unique = []
    for name in names:
        if name not in seen:
            seen[name] = 0
            unique.append(name)
        else:
            seen[name] += 1
            unique.append(f"{name}_{seen[name]}")
    return unique


def clean_feature_name(name):
    return str(name).replace("continuous__", "").replace("binary__", "").replace("remainder__", "")


def white_plot(fig):
    fig.update_layout(template="plotly_white")
    fig.show()


def regression_metrics(y, pred, config=CONFIG):
    """Regression metrics for recipe-rating predictions."""
    low_clip = config.min_target_value if config.filter_low_ratings else 1.0
    pred = np.clip(pred, low_clip, config.max_target_value)
    return {
        "MAE": mean_absolute_error(y, pred),
        "RMSE": np.sqrt(mean_squared_error(y, pred)),
        "R2": r2_score(y, pred),
        "Within_0.10": np.mean(np.abs(y - pred) <= 0.10),
        "Within_0.25": np.mean(np.abs(y - pred) <= 0.25),
        "Within_0.50": np.mean(np.abs(y - pred) <= 0.50),
    }


def compute_vif(data, features):
    """Manual VIF calculation using linear regression on numeric continuous variables."""
    if len(features) < 2:
        return pd.DataFrame(columns=["feature", "VIF"])

    X = data[features].replace([np.inf, -np.inf], np.nan)
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))
    X = X.loc[:, X.nunique() > 1]

    rows = []
    for col in X.columns:
        y = X[col]
        X_other = X.drop(columns=col)
        if X_other.shape[1] == 0:
            vif = 1.0
        else:
            r2 = LinearRegression().fit(X_other, y).score(X_other, y)
            vif = np.inf if r2 >= 0.999999 else 1 / (1 - r2)
        rows.append([col, vif])

    return pd.DataFrame(rows, columns=["feature", "VIF"]).sort_values("VIF", ascending=False)


def make_rating_bins(y, q=5):
    """Create stratification bins for train/test split when possible."""
    try:
        bins = pd.qcut(y, q=q, duplicates="drop")
        return bins if bins.value_counts().min() >= 2 else None
    except Exception:
        return None


class IQRClipper(BaseEstimator, TransformerMixin):
    """Train-fitted IQR winsorizer to avoid using the test distribution during preprocessing."""
    def __init__(self, k=1.5, enabled=True):
        self.k = k
        self.enabled = enabled

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        q1 = np.nanpercentile(X, 25, axis=0)
        q3 = np.nanpercentile(X, 75, axis=0)
        iqr = q3 - q1
        low = q1 - self.k * iqr
        high = q3 + self.k * iqr
        self.low_ = np.where(np.isfinite(low), low, -np.inf)
        self.high_ = np.where(np.isfinite(high), high, np.inf)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        if not self.enabled:
            return X
        return np.clip(X, self.low_, self.high_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


class RatingBandLogisticRegressor(BaseEstimator, RegressorMixin):
    """
    Logistic-regression challenger adapted to the continuous rating target.

    LogisticRegression is a classifier. This wrapper bins the rating target into ordered rating bands,
    trains LogisticRegression on those bands, then converts predicted class probabilities back into
    continuous rating estimates using each band's average training rating.
    """
    def __init__(self, bins=None, C=1.0, max_iter=1000, random_state=None, min_target_value=4.0, max_target_value=5.0):
        self.bins = bins
        self.C = C
        self.max_iter = max_iter
        self.random_state = random_state
        self.min_target_value = min_target_value
        self.max_target_value = max_target_value

    def fit(self, X, y, sample_weight=None):
        y = pd.Series(y).astype(float).reset_index(drop=True)

        if self.bins is None:
            bins = np.array([self.min_target_value, 4.25, 4.50, 4.75, self.max_target_value], dtype=float)
        else:
            bins = np.asarray(self.bins, dtype=float)

        bins = bins.copy()
        bins[0] = min(bins[0], float(np.nanmin(y)))
        bins[-1] = max(bins[-1], float(np.nanmax(y))) + 1e-8
        self.bin_edges_ = bins

        y_class = pd.cut(y, bins=self.bin_edges_, labels=False, include_lowest=True).astype(int)
        self.class_rating_values_ = y.groupby(y_class).mean().to_dict()

        if y_class.nunique() < 2:
            self.constant_prediction_ = float(y.mean())
            self.model_ = None
            return self

        self.constant_prediction_ = None
        self.model_ = LogisticRegression(
            C=self.C,
            max_iter=self.max_iter,
            solver="lbfgs",
            random_state=self.random_state
        )
        self.model_.fit(X, y_class, sample_weight=sample_weight)
        return self

    def predict(self, X):
        if self.model_ is None:
            return np.repeat(self.constant_prediction_, X.shape[0])

        proba = self.model_.predict_proba(X)
        class_values = np.array([
            self.class_rating_values_.get(int(cls), (self.bin_edges_[int(cls)] + self.bin_edges_[int(cls) + 1]) / 2)
            for cls in self.model_.classes_
        ])
        return proba @ class_values

    @property
    def coef_(self):
        if self.model_ is None:
            raise AttributeError("The fallback constant model has no coefficients.")
        return self.model_.coef_


## 2 — Feature engineering components

In [91]:
@dataclass
class RecipeDataset:
    data: pd.DataFrame
    continuous_features: List[str]
    binary_features: List[str]
    ingredient_features: List[str]
    ingredient_group_features: List[str]
    source_category_features: List[str]
    feature_component_map: Dict[str, str]
    filter_report: Dict[str, Any]


class DataLoader:
    """Loads the raw recipe tables and removes columns that should not be modeled."""
    def __init__(self, config: RecipeModelConfig):
        self.config = config

    def load(self) -> Tuple[pd.DataFrame, pd.DataFrame]:
        nrows = self.config.quick_run_rows if self.config.quick_run else None
        allrecipes_df = pd.read_csv(self.config.allrecipes_file, nrows=nrows)
        ingredients_df = pd.read_csv(self.config.ingredients_file, nrows=nrows)

        # Author variables are intentionally excluded from the modeling framework.
        allrecipes_df = allrecipes_df.drop(columns=list(self.config.excluded_raw_columns), errors="ignore")
        ingredients_df = ingredients_df.drop(columns=list(self.config.excluded_raw_columns), errors="ignore")

        return allrecipes_df, ingredients_df


class NutritionFeatureBuilder:
    """Cleans nutrition fields and creates nutrition-derived features."""
    nutrition_map = {
        "nutrition_calories": "calories",
        "nutrition_carbs": "carbs",
        "nutrition_fat": "fat",
        "nutrition_protein": "protein",
    }

    nutrition_cols = ["calories", "carbs", "fat", "protein"]
    engineered_features = [
        "calories", "carbs", "fat", "protein",
        "log_calories", "log_carbs", "log_fat", "log_protein",
        "protein_per_calorie", "fat_per_calorie", "carbs_per_calorie", "calories_per_ingredient"
    ]

    def clean_raw_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        for raw, clean in self.nutrition_map.items():
            if raw in df.columns:
                df[clean] = df[raw].apply(extract_number)
        return df

    def add_engineered_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        for col in self.nutrition_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
                df[f"log_{col}"] = np.log1p(df[col].clip(lower=0))

        if {"protein", "calories"}.issubset(df.columns):
            df["protein_per_calorie"] = safe_divide(df["protein"], df["calories"])
        if {"fat", "calories"}.issubset(df.columns):
            df["fat_per_calorie"] = safe_divide(df["fat"], df["calories"])
        if {"carbs", "calories"}.issubset(df.columns):
            df["carbs_per_calorie"] = safe_divide(df["carbs"], df["calories"])
        if {"calories", "ingredient_count"}.issubset(df.columns):
            df["calories_per_ingredient"] = safe_divide(df["calories"], df["ingredient_count"])

        return df


class IngredientFeatureBuilder:
    """
    Aggregates the long ingredient table and creates two levels of ingredient features.

    Level 1 — ingredient classes:
        Groups many ingredient names into interpretable classes such as dairy, vegetables, grains, etc.
    Level 2 — individual ingredient flags:
        Keeps the top N individual ingredients as detailed binary flags.
    """
    def __init__(self, config: RecipeModelConfig):
        self.config = config
        self.top_ingredients_: List[str] = []
        self.ingredient_features_: List[str] = []
        self.ingredient_group_features_: List[str] = []
        self.ingredient_group_count_features_: List[str] = []
        self.ingredient_group_presence_features_: List[str] = []
        self.ingredient_group_metric_features_: List[str] = []
        self.ingredient_group_lookup_: Dict[str, str] = {}

    def _clean_ingredient_text(self, ingredients_df: pd.DataFrame) -> pd.DataFrame:
        df = ingredients_df.copy()
        for col in ["ingredient_canonical", "ingredient_clean", "ingredient_raw"]:
            if col not in df.columns:
                df[col] = ""
            df[col] = df[col].fillna("").astype(str).str.lower().str.strip()

        df["ingredient_text_for_group"] = (
            df["ingredient_canonical"] + " " + df["ingredient_clean"] + " " + df["ingredient_raw"]
        ).str.replace(r"\s+", " ", regex=True).str.strip()
        return df

    def _add_ingredient_group_flags(self, ingredients_df: pd.DataFrame) -> pd.DataFrame:
        df = ingredients_df.copy()
        if not self.config.use_ingredient_group_features:
            return df

        for group_name, patterns in self.config.ingredient_group_patterns.items():
            safe_group = make_safe_column_name(group_name, prefix="inggrp_").replace("inggrp_inggrp_", "inggrp_")
            raw_flag = f"__{safe_group}_match"
            regex = "|".join(f"(?:{p})" for p in patterns)
            df[raw_flag] = df["ingredient_text_for_group"].str.contains(regex, regex=True, na=False)
            self.ingredient_group_lookup_[raw_flag] = safe_group

        return df

    def _build_ingredient_group_features(self, recipe_summary: pd.DataFrame, ingredients_df: pd.DataFrame) -> pd.DataFrame:
        if not self.config.use_ingredient_group_features or not self.ingredient_group_lookup_:
            return recipe_summary

        dedup_cols = ["recipe_id", "ingredient_canonical"] + list(self.ingredient_group_lookup_.keys())
        dedup = ingredients_df[dedup_cols].drop_duplicates()

        group_counts = (
            dedup
            .groupby("recipe_id", as_index=False)[list(self.ingredient_group_lookup_.keys())]
            .sum()
            .rename(columns={raw: f"{safe}_count" for raw, safe in self.ingredient_group_lookup_.items()})
        )

        count_cols = [f"{safe}_count" for safe in self.ingredient_group_lookup_.values()]
        presence_cols = [col.replace("_count", "_present") for col in count_cols]

        for count_col, present_col in zip(count_cols, presence_cols):
            group_counts[present_col] = (group_counts[count_col] > 0).astype(int)

        recipe_summary = recipe_summary.merge(group_counts, on="recipe_id", how="left")
        recipe_summary[count_cols + presence_cols] = recipe_summary[count_cols + presence_cols].fillna(0)
        recipe_summary[count_cols] = recipe_summary[count_cols].astype(float)
        recipe_summary[presence_cols] = recipe_summary[presence_cols].astype(int)

        recipe_summary["ingredient_class_count"] = (recipe_summary[count_cols] > 0).sum(axis=1)
        recipe_summary["ingredient_class_diversity"] = safe_divide(recipe_summary["ingredient_class_count"], recipe_summary["ingredient_count"])
        recipe_summary["largest_ingredient_class_share"] = safe_divide(recipe_summary[count_cols].max(axis=1), recipe_summary["ingredient_count"])

        self.ingredient_group_count_features_ = count_cols
        self.ingredient_group_presence_features_ = presence_cols
        self.ingredient_group_metric_features_ = [
            "ingredient_class_count",
            "ingredient_class_diversity",
            "largest_ingredient_class_share"
        ]
        self.ingredient_group_features_ = count_cols + presence_cols + self.ingredient_group_metric_features_
        return recipe_summary

    def _build_individual_ingredient_flags(self, recipe_summary: pd.DataFrame, ingredients_df: pd.DataFrame) -> pd.DataFrame:
        if not self.config.use_individual_ingredient_flags:
            self.ingredient_features_ = []
            return recipe_summary

        top_ingredients = (
            ingredients_df.loc[ingredients_df["ingredient_canonical"] != "", "ingredient_canonical"]
            .value_counts()
            .head(self.config.top_n_ingredients)
            .index
            .tolist()
        )
        self.top_ingredients_ = top_ingredients

        if not top_ingredients:
            self.ingredient_features_ = []
            return recipe_summary

        flag_source = ingredients_df[ingredients_df["ingredient_canonical"].isin(top_ingredients)].copy()
        ingredient_flags = (
            flag_source
            .assign(value=1)
            .pivot_table(
                index="recipe_id",
                columns="ingredient_canonical",
                values="value",
                aggfunc="max",
                fill_value=0
            )
            .reset_index()
        )

        raw_flag_cols = [c for c in ingredient_flags.columns if c != "recipe_id"]
        safe_flag_cols = make_unique([make_safe_column_name(c, prefix="ing_") for c in raw_flag_cols])
        ingredient_flags = ingredient_flags.rename(columns=dict(zip(raw_flag_cols, safe_flag_cols)))

        recipe_summary = recipe_summary.merge(ingredient_flags, on="recipe_id", how="left")
        recipe_summary[safe_flag_cols] = recipe_summary[safe_flag_cols].fillna(0).astype(int)
        self.ingredient_features_ = safe_flag_cols
        return recipe_summary

    def build_recipe_level_table(self, ingredients_df: pd.DataFrame) -> pd.DataFrame:
        ingredients_df = self._clean_ingredient_text(ingredients_df)
        ingredients_df = self._add_ingredient_group_flags(ingredients_df)

        agg_dict = {"ingredient_count": ("ingredient_canonical", lambda x: x.replace("", np.nan).nunique())}
        for col in ["rating_value", "rating_count", "calories", "carbs", "fat", "protein"]:
            if col in ingredients_df.columns:
                agg_dict[col] = (col, "first")

        recipe_summary = (
            ingredients_df
            .groupby(["recipe_id", "title", "url"], as_index=False)
            .agg(**agg_dict)
        )

        recipe_summary = self._build_ingredient_group_features(recipe_summary, ingredients_df)
        recipe_summary = self._build_individual_ingredient_flags(recipe_summary, ingredients_df)
        return recipe_summary


class RecipeComplexityBuilder:
    """Creates recipe complexity features from text, directions, ingredient lists, and durations."""
    engineered_features = [
        "ingredient_count", "log_ingredient_count",
        "prep_minutes", "cook_minutes", "total_minutes",
        "log_prep_minutes", "log_cook_minutes", "log_total_minutes",
        "description_word_count", "title_word_count", "direction_step_count", "raw_ingredient_item_count",
        "category_count", "cuisine_count", "minutes_per_step", "ingredients_per_step", "steps_per_ingredient",
        "total_minutes_per_ingredient", "description_words_per_step"
    ]

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        df["description"] = df["description"].fillna("").astype(str).str.lower() if "description" in df.columns else ""
        df["title"] = df["title"].fillna("").astype(str).str.lower() if "title" in df.columns else ""

        df["description_word_count"] = df["description"].str.split().str.len()
        df["title_word_count"] = df["title"].str.split().str.len()

        if "directions" in df.columns:
            df["direction_step_count"] = df["directions"].apply(count_items)
        if "ingredients" in df.columns:
            df["raw_ingredient_item_count"] = df["ingredients"].apply(count_items)
        if "category" in df.columns:
            df["category_count"] = df["category"].apply(lambda x: len(safe_literal_list(x)))
        if "cuisine" in df.columns:
            df["cuisine_count"] = df["cuisine"].apply(lambda x: len(safe_literal_list(x)))

        for raw, clean in [("prep_time", "prep_minutes"), ("cook_time", "cook_minutes"), ("total_time", "total_minutes")]:
            if raw in df.columns:
                df[clean] = df[raw].apply(parse_minutes)
                df[f"log_{clean}"] = np.log1p(df[clean].clip(lower=0))

        if "ingredient_count" in df.columns:
            df["ingredient_count"] = pd.to_numeric(df["ingredient_count"], errors="coerce")
            df["log_ingredient_count"] = np.log1p(df["ingredient_count"])

        if {"total_minutes", "direction_step_count"}.issubset(df.columns):
            df["minutes_per_step"] = safe_divide(df["total_minutes"], df["direction_step_count"])
        if {"ingredient_count", "direction_step_count"}.issubset(df.columns):
            df["ingredients_per_step"] = safe_divide(df["ingredient_count"], df["direction_step_count"])
            df["steps_per_ingredient"] = safe_divide(df["direction_step_count"], df["ingredient_count"])
        if {"total_minutes", "ingredient_count"}.issubset(df.columns):
            df["total_minutes_per_ingredient"] = safe_divide(df["total_minutes"], df["ingredient_count"])
        if {"description_word_count", "direction_step_count"}.issubset(df.columns):
            df["description_words_per_step"] = safe_divide(df["description_word_count"], df["direction_step_count"])

        return df


class MetadataFeatureBuilder:
    """Merges selected metadata and creates time/source-category features."""
    metadata_cols = [
        "url", "description", "category", "cuisine", "source_category",
        "prep_time", "cook_time", "total_time", "directions", "ingredients", "date_published"
    ]
    engineered_features = ["publication_year", "publication_month", "category_count", "cuisine_count"]

    def __init__(self, top_n_source_categories=15):
        self.top_n_source_categories = top_n_source_categories
        self.source_category_features_: List[str] = []

    def merge_metadata(self, recipe_summary: pd.DataFrame, allrecipes_df: pd.DataFrame) -> pd.DataFrame:
        meta_cols = [c for c in self.metadata_cols if c in allrecipes_df.columns]
        return recipe_summary.merge(
            allrecipes_df[meta_cols].drop_duplicates("url"),
            on="url",
            how="left"
        )

    def add_publication_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        if "date_published" in df.columns:
            d = pd.to_datetime(df["date_published"], errors="coerce", utc=True)
            df["publication_year"] = d.dt.year
            df["publication_month"] = d.dt.month
        return df

    def add_source_category_flags(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        if "source_category" not in df.columns:
            self.source_category_features_ = []
            return df

        s = df["source_category"].fillna("missing").astype(str).str.lower().str.strip()
        top_values = s.value_counts().head(self.top_n_source_categories).index.tolist()
        source_features = make_unique([make_safe_column_name(value, prefix="src_") for value in top_values])

        for value, new_col in zip(top_values, source_features):
            df[new_col] = (s == value).astype(int)

        self.source_category_features_ = source_features
        return df


class RecipeDatasetBuilder:
    """Orchestrates all feature components and returns the modeling dataset."""
    def __init__(self, config: RecipeModelConfig):
        self.config = config
        self.nutrition = NutritionFeatureBuilder()
        self.ingredients = IngredientFeatureBuilder(config)
        self.complexity = RecipeComplexityBuilder()
        self.metadata = MetadataFeatureBuilder(config.top_n_source_categories)

    def build(self, allrecipes_df: pd.DataFrame, ingredients_df: pd.DataFrame) -> RecipeDataset:
        allrecipes_df = self.nutrition.clean_raw_columns(allrecipes_df)
        ingredients_df = self.nutrition.clean_raw_columns(ingredients_df)

        recipe_summary = self.ingredients.build_recipe_level_table(ingredients_df)
        recipe_summary = self.metadata.merge_metadata(recipe_summary, allrecipes_df)
        recipe_summary = self.complexity.transform(recipe_summary)
        recipe_summary = self.metadata.add_publication_features(recipe_summary)
        recipe_summary = self.metadata.add_source_category_flags(recipe_summary)
        recipe_summary = self.nutrition.add_engineered_features(recipe_summary)

        recipe_summary = self._convert_core_numeric(recipe_summary)
        recipe_summary, filter_report = self._filter_modeling_rows(recipe_summary)

        continuous_features = self._continuous_feature_candidates(recipe_summary)
        if self.config.use_rating_count_as_feature and "rating_count" in recipe_summary.columns:
            continuous_features += ["rating_count"]
            if "log_rating_count" in recipe_summary.columns:
                continuous_features += ["log_rating_count"]

        binary_features = self.ingredients.ingredient_group_presence_features_ + self.ingredients.ingredient_features_ + self.metadata.source_category_features_
        binary_features = [c for c in binary_features if c in recipe_summary.columns]
        continuous_features = [c for c in continuous_features if c in recipe_summary.columns]

        feature_component_map = self._build_feature_component_map(continuous_features, binary_features)

        return RecipeDataset(
            data=recipe_summary,
            continuous_features=continuous_features,
            binary_features=binary_features,
            ingredient_features=[c for c in self.ingredients.ingredient_features_ if c in recipe_summary.columns],
            ingredient_group_features=[c for c in self.ingredients.ingredient_group_features_ if c in recipe_summary.columns],
            source_category_features=[c for c in self.metadata.source_category_features_ if c in recipe_summary.columns],
            feature_component_map=feature_component_map,
            filter_report=filter_report
        )

    def _convert_core_numeric(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        for col in ["rating_value", "rating_count", "calories", "carbs", "fat", "protein", "ingredient_count"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        if "rating_count" in df.columns:
            df["log_rating_count"] = np.log1p(df["rating_count"].fillna(0).clip(lower=0))

        return df

    def _filter_modeling_rows(self, df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
        before_shape = df.shape
        target = self.config.target

        target_mask = df[target].between(1, self.config.max_target_value)
        count_mask = df["rating_count"].fillna(0).ge(self.config.min_rating_count) if "rating_count" in df.columns else True
        df = df[target_mask & count_mask].copy()

        if self.config.filter_low_ratings:
            before_low_rating_filter = df.shape[0]
            df = df[df[target] >= self.config.min_target_value].copy()
            removed_low = before_low_rating_filter - df.shape[0]
        else:
            removed_low = 0

        report = {
            "before_shape": before_shape,
            "after_shape": df.shape,
            "removed_low_rating_rows": removed_low,
        }
        return df, report

    def _continuous_feature_candidates(self, df: pd.DataFrame) -> List[str]:
        candidates = [
            # Ingredient class features
            "ingredient_class_count", "ingredient_class_diversity", "largest_ingredient_class_share",
            *self.ingredients.ingredient_group_count_features_,

            # Recipe complexity features
            "ingredient_count", "log_ingredient_count",
            "prep_minutes", "cook_minutes", "total_minutes",
            "log_prep_minutes", "log_cook_minutes", "log_total_minutes",
            "description_word_count", "title_word_count", "direction_step_count", "raw_ingredient_item_count",
            "category_count", "cuisine_count", "minutes_per_step", "ingredients_per_step", "steps_per_ingredient",
            "total_minutes_per_ingredient", "description_words_per_step",

            # Metadata timing features
            "publication_year", "publication_month",

            # Nutrition features
            "calories", "carbs", "fat", "protein",
            "log_calories", "log_carbs", "log_fat", "log_protein",
            "protein_per_calorie", "fat_per_calorie", "carbs_per_calorie", "calories_per_ingredient"
        ]
        return [c for c in make_unique(candidates) if c in df.columns]

    def _build_feature_component_map(self, continuous_features: List[str], binary_features: List[str]) -> Dict[str, str]:
        feature_component_map: Dict[str, str] = {}

        ingredient_class_features = set(
            self.ingredients.ingredient_group_count_features_
            + self.ingredients.ingredient_group_presence_features_
            + self.ingredients.ingredient_group_metric_features_
        )
        individual_ingredient_features = set(self.ingredients.ingredient_features_)
        source_category_features = set(self.metadata.source_category_features_)
        nutrition_features = set(self.nutrition.engineered_features)
        complexity_features = set(self.complexity.engineered_features)
        metadata_features = {"publication_year", "publication_month", "category_count", "cuisine_count"}
        popularity_features = {"rating_count", "log_rating_count"}

        for feature in continuous_features + binary_features:
            if feature in ingredient_class_features:
                component = "ingredient_classes"
            elif feature in individual_ingredient_features:
                component = "individual_ingredients"
            elif feature in nutrition_features:
                component = "nutrition"
            elif feature in complexity_features:
                component = "recipe_complexity"
            elif feature in metadata_features or feature in source_category_features:
                component = "metadata_context"
            elif feature in popularity_features:
                component = "popularity_signal"
            else:
                component = "other"
            feature_component_map[feature] = component

        return feature_component_map


## 3 — Data quality controls and modeling components

In [92]:
class FeatureControls:
    """Applies transparent feature quality controls before modeling."""
    def __init__(self, config: RecipeModelConfig):
        self.config = config

    def apply(
        self,
        df: pd.DataFrame,
        continuous_features: List[str],
        binary_features: List[str]
    ) -> Tuple[pd.DataFrame, List[str], List[str], Dict[str, pd.DataFrame]]:
        df = df.copy()
        target = self.config.target
        features = continuous_features + binary_features
        diagnostics = {}

        missing_table = (
            df[features + [target]]
            .isna()
            .mean()
            .sort_values(ascending=False)
            .reset_index()
        )
        missing_table.columns = ["feature", "missing_rate"]
        diagnostics["missing_table"] = missing_table

        if self.config.use_missing_control:
            high_missing = missing_table.query("missing_rate > @self.config.max_missing_rate and feature != @target")["feature"].tolist()
            if high_missing:
                if self.config.control_mode == "stop":
                    raise ValueError("Missingness threshold exceeded.")
                if self.config.control_mode == "filter":
                    continuous_features = [c for c in continuous_features if c not in high_missing]
                    binary_features = [c for c in binary_features if c not in high_missing]

        features = continuous_features + binary_features

        variance_table = pd.DataFrame({
            "feature": features,
            "n_unique": [df[c].nunique(dropna=True) for c in features]
        }).sort_values("n_unique")
        diagnostics["variance_table"] = variance_table

        if self.config.use_low_variance_control:
            low_variance_features = variance_table.query("n_unique < @self.config.min_unique_values")["feature"].tolist()
            if low_variance_features:
                if self.config.control_mode == "stop":
                    raise ValueError("Low-variance feature threshold exceeded.")
                if self.config.control_mode == "filter":
                    continuous_features = [c for c in continuous_features if c not in low_variance_features]
                    binary_features = [c for c in binary_features if c not in low_variance_features]

        outlier_table = self._outlier_diagnostics(df, continuous_features)
        diagnostics["outlier_table"] = outlier_table

        if (
            self.config.use_outlier_control
            and self.config.control_mode == "filter"
            and self.config.outlier_action == "remove_rows"
            and len(outlier_table) > 0
        ):
            keep_mask = pd.Series(True, index=df.index)
            for _, row in outlier_table.iterrows():
                col = row["feature"]
                keep_mask &= df[col].between(row["low_limit"], row["high_limit"]) | df[col].isna()
            df = df.loc[keep_mask].copy()

        corr_table, corr_matrix = self._correlation_control(df, continuous_features)
        diagnostics["high_corr_table"] = corr_table
        diagnostics["corr_matrix"] = corr_matrix

        if self.config.use_correlation_control and len(corr_table) > 0:
            if self.config.control_mode == "stop":
                raise ValueError("Correlation threshold exceeded.")
            if self.config.control_mode == "filter":
                drop_corr = corr_table["suggested_drop"].unique().tolist()
                continuous_features = [c for c in continuous_features if c not in drop_corr]
                binary_features = [c for c in binary_features if c not in drop_corr]

        vif_features = [c for c in continuous_features if c in df.columns]
        vif_before = compute_vif(df, vif_features)
        diagnostics["vif_before"] = vif_before

        if self.config.use_vif_control:
            if self.config.control_mode == "stop" and len(vif_before) and (vif_before["VIF"] > self.config.vif_threshold).any():
                raise ValueError("VIF threshold exceeded.")
            if self.config.control_mode == "filter":
                while True:
                    vif_table = compute_vif(df, vif_features)
                    if len(vif_table) == 0 or vif_table.iloc[0]["VIF"] <= self.config.vif_threshold:
                        break
                    drop_col = vif_table.iloc[0]["feature"]
                    vif_features.remove(drop_col)
                    continuous_features = [c for c in continuous_features if c != drop_col]

        diagnostics["vif_after"] = compute_vif(df, [c for c in continuous_features if c in df.columns])
        return df, continuous_features, binary_features, diagnostics

    def _outlier_diagnostics(self, df, continuous_features):
        outlier_rows = []
        for col in continuous_features:
            q1 = df[col].quantile(0.25)
            q3 = df[col].quantile(0.75)
            iqr = q3 - q1
            low = q1 - self.config.iqr_k * iqr
            high = q3 + self.config.iqr_k * iqr
            rate = ((df[col] < low) | (df[col] > high)).mean()
            outlier_rows.append([col, low, high, rate])
        return pd.DataFrame(outlier_rows, columns=["feature", "low_limit", "high_limit", "outlier_rate"]).sort_values("outlier_rate", ascending=False)

    def _correlation_control(self, df, continuous_features):
        corr_features = [c for c in continuous_features if c in df.columns and df[c].nunique(dropna=True) > 1]
        corr_matrix = df[corr_features + [self.config.target]].corr() if corr_features else pd.DataFrame()
        missing_lookup = df[corr_features].isna().mean().to_dict() if corr_features else {}
        high_corr_pairs = []

        for i, col1 in enumerate(corr_features):
            for col2 in corr_features[i + 1:]:
                corr = corr_matrix.loc[col1, col2]
                if pd.notna(corr) and abs(corr) >= self.config.corr_threshold:
                    miss1 = missing_lookup.get(col1, 0)
                    miss2 = missing_lookup.get(col2, 0)
                    drop_col = col1 if miss1 > miss2 else col2
                    high_corr_pairs.append([col1, col2, corr, miss1, miss2, drop_col])

        corr_table = pd.DataFrame(
            high_corr_pairs,
            columns=["feature_1", "feature_2", "correlation", "missing_1", "missing_2", "suggested_drop"]
        )
        return corr_table, corr_matrix


class ModelTrainer:
    """Builds preprocessing, trains models, and fixes Gradient Boosting as the champion."""
    def __init__(self, config: RecipeModelConfig):
        self.config = config

    def make_preprocess(self, continuous_features, binary_features):
        continuous_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clipper", IQRClipper(
                k=self.config.iqr_k,
                enabled=self.config.use_outlier_control and self.config.outlier_action == "winsorize"
            )),
            ("scaler", RobustScaler())
        ])

        binary_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent"))
        ])

        return ColumnTransformer(
            transformers=[
                ("continuous", continuous_pipeline, continuous_features),
                ("binary", binary_pipeline, binary_features),
            ],
            remainder="drop",
            verbose_feature_names_out=True
        )

    def train_test_data(self, df, features):
        X = df[features].copy()
        y = df[self.config.target].copy()

        sample_weight = (
            np.log1p(df["rating_count"].fillna(0)).clip(lower=1)
            if self.config.use_sample_weights and "rating_count" in df.columns
            else None
        )

        stratify_bins = make_rating_bins(y) if self.config.use_stratified_split else None

        if sample_weight is not None:
            X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
                X, y, sample_weight,
                test_size=self.config.test_size,
                random_state=self.config.random_state,
                stratify=stratify_bins
            )
        else:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y,
                test_size=self.config.test_size,
                random_state=self.config.random_state,
                stratify=stratify_bins
            )
            w_train, w_test = None, None

        return X_train, X_test, y_train, y_test, w_train, w_test

    def build_models(self):
        return {
            "Baseline — Mean": DummyRegressor(strategy="mean"),

            self.config.champion_model_name: GradientBoostingRegressor(
                n_estimators=200,
                learning_rate=0.04,
                max_depth=3,
                min_samples_leaf=10,
                random_state=self.config.random_state
            ),

            "Challenger 1 — Ridge": Ridge(alpha=1.0),

            "Challenger 2 — Logistic Regression": RatingBandLogisticRegressor(
                C=1.0,
                max_iter=1000,
                random_state=self.config.random_state,
                min_target_value=self.config.min_target_value if self.config.filter_low_ratings else 1.0,
                max_target_value=self.config.max_target_value
            ),

            "Challenger 3 — Random Forest Regressor": RandomForestRegressor(
                n_estimators=250,
                min_samples_leaf=3,
                max_features="sqrt",
                random_state=self.config.random_state,
                n_jobs=self.config.n_jobs
            )
        }

    def fit_evaluate(self, preprocess, X_train, X_test, y_train, y_test, w_train=None):
        rmse_scorer = make_scorer(
            lambda yt, yp: np.sqrt(mean_squared_error(
                yt,
                np.clip(
                    yp,
                    self.config.min_target_value if self.config.filter_low_ratings else 1.0,
                    self.config.max_target_value
                )
            )),
            greater_is_better=False
        )

        results = []
        fitted_models = {}
        models = self.build_models()

        for name, model in models.items():
            print(f"Training {name}...")
            pipe = Pipeline([
                ("preprocess", clone(preprocess)),
                ("model", model)
            ])

            row = {"model": name}

            if self.config.run_cross_validation:
                cv = KFold(n_splits=self.config.cv_folds, shuffle=True, random_state=self.config.random_state)
                cv_scores = cross_validate(
                    pipe,
                    X_train,
                    y_train,
                    cv=cv,
                    scoring={"MAE": "neg_mean_absolute_error", "RMSE": rmse_scorer, "R2": "r2"},
                    n_jobs=self.config.n_jobs
                )
                row["cv_MAE"] = -cv_scores["test_MAE"].mean()
                row["cv_RMSE"] = -cv_scores["test_RMSE"].mean()
                row["cv_R2"] = cv_scores["test_R2"].mean()

            try:
                pipe.fit(X_train, y_train, model__sample_weight=w_train)
            except TypeError:
                pipe.fit(X_train, y_train)

            train_pred = pipe.predict(X_train)
            test_pred = pipe.predict(X_test)

            for k, v in regression_metrics(y_train, train_pred, self.config).items():
                row["train_" + k] = v
            for k, v in regression_metrics(y_test, test_pred, self.config).items():
                row["test_" + k] = v

            results.append(row)
            fitted_models[name] = pipe

        results_df = pd.DataFrame(results).sort_values("test_R2", ascending=False)
        champion_model = fitted_models[self.config.champion_model_name]
        champion_row = results_df[results_df["model"] == self.config.champion_model_name].iloc[0]

        return results_df, fitted_models, champion_model, champion_row


# 4 — Load data

In [93]:
loader = DataLoader(CONFIG)
allrecipes_df, ingredients_df = loader.load()

print("ALLRECIPES DATASET")
print("Shape:", allrecipes_df.shape)
display(allrecipes_df.head())

print("\nINGREDIENTS DATASET")
print("Shape:", ingredients_df.shape)
display(ingredients_df.head())


ALLRECIPES DATASET
Shape: (14438, 24)


,category,cook_time,cuisine,date_published,description,directions,ingredients,letter,nutrition_calories,nutrition_carbs,...,rating_count,rating_text,rating_value,review_count,source_category,source_category_url,title,total_time,url,yield
0,"['Dinner', 'Entree']",PT15M,['American'],2025-06-05T09:15:00-04:00,These 3-ingredient air fryer everything bagel ...,"[""Gather all ingredients. Preheat an air fryer...","[""1 1/4 pound fresh chicken tenders"", ""1 table...",A,366 kcal,22 g,...,8.0,NaN,4.6,NaN,Air Fryer Recipes,https://www.allrecipes.com/recipes/23070/every...,3-Ingredient Air Fryer Everything Bagel Chicke...,PT20M,https://www.allrecipes.com/3-ingredient-air-fr...,5
1,"['Appetizer', 'Dinner']",PT10M,['American'],2024-02-23T19:16:13-05:00,These 4 ingredient air fryer pepper poppers— c...,"[""Preheat an air fryer to 390 degrees F (199 d...","[""1 bell pepper, any color"", ""8 ounces cream c...",A,244 kcal,6 g,...,0.0,Be the first to rate & review!,NaN,0.0,Air Fryer Recipes,https://www.allrecipes.com/recipes/23070/every...,4 Ingredient Air Fryer Pepper Poppers,PT20M,https://www.allrecipes.com/4-ingredient-air-fr...,4
2,"['Appetizer', 'Dinner']",PT10M,['Mexican'],2025-02-01T06:00:00-05:00,These air fried tossed taquitos are frozen chi...,"[""Heat an air fryer to 400 degrees F (200 degr...","[""24 frozen chicken taquitos"", ""2 Tablespoons ...",A,534 kcal,57 g,...,0.0,Be the first to rate & review!,NaN,0.0,Air Fryer Recipes,https://www.allrecipes.com/recipes/23070/every...,Air Fried Tossed Taquitos,PT15M,https://www.allrecipes.com/air-fried-tossed-ta...,6
3,"['Snack', 'Lunch', 'Entree']",PT5M,['American'],2026-04-08T17:15:00-04:00,These air-fried 2-ingredient mozzarella sticks...,"[""Place a cheese stick at the bottom third of ...","[""8 string cheese"", ""8 6 inch flour tortillas""...",A,395 kcal,30 g,...,4.0,NaN,4.5,NaN,Air Fryer Recipes,https://www.allrecipes.com/recipes/23070/every...,Air Fryer 2-Ingredient Mozzarella Sticks,PT20M,https://www.allrecipes.com/air-fryer-2-ingredi...,4
4,"['Dinner', 'Side Dish']",PT15M,['American'],2024-11-07T21:00:00-05:00,"These air fryer baked yams free up the oven, w...","[""Preheat an air fryer to 400 degrees F (200 d...","[""1 yam"", ""1/2 teaspoon olive oil""]",A,283 kcal,62 g,...,0.0,Be the first to rate & review!,NaN,0.0,Air Fryer Recipes,https://www.allrecipes.com/recipes/23070/every...,Air Fryer Baked Yams,PT20M,https://www.allrecipes.com/air-fryer-baked-yam...,1



INGREDIENTS DATASET
Shape: (139238, 19)


,recipe_id,title,ingredient_index,ingredient_raw,ingredient_clean,ingredient_canonical,rating_value,rating_value_num,rating_count,rating_count_num,nutrition_calories,nutrition_calories_num,nutrition_carbs,nutrition_carbs_num,nutrition_fat,nutrition_fat_num,nutrition_protein,nutrition_protein_num,url
0,0,3-Ingredient Air Fryer Everything Bagel Chicke...,1,1 1/4 pound fresh chicken tenders,chicken tenders,chicken tender,4.6,4.6,8.0,8.0,366 kcal,366.0,22 g,22.0,22 g,22.0,21 g,21.0,https://www.allrecipes.com/3-ingredient-air-fr...
1,0,3-Ingredient Air Fryer Everything Bagel Chicke...,2,1 tablespoon olive oil,olive oil,olive oil,4.6,4.6,8.0,8.0,366 kcal,366.0,22 g,22.0,22 g,22.0,21 g,21.0,https://www.allrecipes.com/3-ingredient-air-fr...
2,0,3-Ingredient Air Fryer Everything Bagel Chicke...,3,1/3 cup everything bagel seasoning,everything bagel seasoning,everything bagel seasoning,4.6,4.6,8.0,8.0,366 kcal,366.0,22 g,22.0,22 g,22.0,21 g,21.0,https://www.allrecipes.com/3-ingredient-air-fr...
3,1,4 Ingredient Air Fryer Pepper Poppers,1,"1 bell pepper, any color",bell pepper,pepper,NaN,NaN,0.0,0.0,244 kcal,244.0,6 g,6.0,23 g,23.0,4 g,4.0,https://www.allrecipes.com/4-ingredient-air-fr...
4,1,4 Ingredient Air Fryer Pepper Poppers,2,"8 ounces cream cheese, softened",cream cheese,cream cheese,NaN,NaN,0.0,0.0,244 kcal,244.0,6 g,6.0,23 g,23.0,4 g,4.0,https://www.allrecipes.com/4-ingredient-air-fr...


# 5 — Build component-based modeling table

In [94]:
dataset_builder = RecipeDatasetBuilder(CONFIG)
dataset = dataset_builder.build(allrecipes_df, ingredients_df)

recipe_summary = dataset.data
continuous_features = dataset.continuous_features
binary_features = dataset.binary_features
features = continuous_features + binary_features
feature_component_map = dataset.feature_component_map.copy()

print("Recipe-level dataset shape before filters:", dataset.filter_report["before_shape"])
print("Rows removed because rating < 4:", dataset.filter_report["removed_low_rating_rows"])
print("Final recipe-level dataset shape:", dataset.filter_report["after_shape"])
print("\nFeature components before controls:")
print("Continuous features:", len(continuous_features))
print("Ingredient-class features:", len(dataset.ingredient_group_features))
print("Individual ingredient flags:", len(dataset.ingredient_features))
print("Source category flags:", len(dataset.source_category_features))
print("Total features before controls:", len(features))

display(recipe_summary.head())


Recipe-level dataset shape before filters: (14396, 157)
Rows removed because rating < 4: 615
Final recipe-level dataset shape: (11196, 157)

Feature components before controls:
Continuous features: 53
Ingredient-class features: 33
Individual ingredient flags: 60
Source category flags: 15
Total features before controls: 143


,recipe_id,title,url,ingredient_count,rating_value,rating_count,calories,carbs,fat,protein,...,src_deviled_eggs,log_calories,log_carbs,log_fat,log_protein,protein_per_calorie,fat_per_calorie,carbs_per_calorie,calories_per_ingredient,log_rating_count
0,0,3-ingredient air fryer everything bagel chicke...,https://www.allrecipes.com/3-ingredient-air-fr...,3,4.6,8.0,366.0,22.0,22.0,21.0,...,0,5.905362,3.135494,3.135494,3.091042,0.057377,0.060109,0.060109,122.000000,2.197225
3,3,air fryer 2-ingredient mozzarella sticks,https://www.allrecipes.com/air-fryer-2-ingredi...,6,4.5,4.0,395.0,30.0,23.0,17.0,...,0,5.981414,3.433987,3.178054,2.890372,0.043038,0.058228,0.075949,65.833333,1.609438
9,9,air fryer chicken parmesan,https://www.allrecipes.com/air-fryer-chicken-p...,12,4.6,7.0,582.0,45.0,18.0,57.0,...,0,6.368187,3.828641,2.944439,4.060443,0.097938,0.030928,0.077320,48.500000,2.079442
11,11,air fryer cinnamon roll bites,https://www.allrecipes.com/air-fryer-cinnamon-...,8,5.0,3.0,281.0,45.0,11.0,3.0,...,0,5.641907,3.828641,2.484907,1.386294,0.010676,0.039146,0.160142,35.125000,1.386294
15,15,air fryer eggplant,https://www.allrecipes.com/air-fryer-eggplant-...,6,4.8,4.0,103.0,10.0,7.0,1.0,...,0,4.644391,2.397895,2.079442,0.693147,0.009709,0.067961,0.097087,17.166667,1.609438


In [95]:
component_overview = (
    pd.DataFrame({"feature": features})
    .assign(component=lambda d: d["feature"].map(feature_component_map).fillna("other"))
    .groupby("component", as_index=False)
    .agg(feature_count=("feature", "count"))
    .sort_values("feature_count", ascending=False)
)

print("Feature components before data controls:")
display(component_overview)

print("\nInitial continuous features:")
print(continuous_features)

print("\nInitial binary features preview:")
print(binary_features[:40], "..." if len(binary_features) > 40 else "")

white_plot(
    px.bar(
        component_overview.sort_values("feature_count"),
        x="feature_count",
        y="component",
        orientation="h",
        title="Number of Features by Component Before Controls"
    )
)


Feature components before data controls:


,component,feature_count
0,individual_ingredients,60
1,ingredient_classes,33
5,recipe_complexity,19
2,metadata_context,17
3,nutrition,12
4,popularity_signal,2



Initial continuous features:
['ingredient_class_count', 'ingredient_class_diversity', 'largest_ingredient_class_share', 'inggrp_meat_poultry_count', 'inggrp_seafood_count', 'inggrp_dairy_count', 'inggrp_eggs_count', 'inggrp_vegetables_count', 'inggrp_fruit_count', 'inggrp_grains_starches_count', 'inggrp_legumes_soy_count', 'inggrp_nuts_seeds_count', 'inggrp_herbs_spices_seasonings_count', 'inggrp_oils_fats_count', 'inggrp_sweeteners_chocolate_count', 'inggrp_sauces_condiments_count', 'inggrp_baking_additives_count', 'inggrp_beverages_alcohol_broth_count', 'ingredient_count', 'log_ingredient_count', 'prep_minutes', 'cook_minutes', 'total_minutes', 'log_prep_minutes', 'log_cook_minutes', 'log_total_minutes', 'description_word_count', 'title_word_count', 'direction_step_count', 'raw_ingredient_item_count', 'category_count', 'cuisine_count', 'minutes_per_step', 'ingredients_per_step', 'steps_per_ingredient', 'total_minutes_per_ingredient', 'description_words_per_step', 'publication_year',

## Ingredient-class feature check

This table shows the new grouped ingredient features. These classes are heuristic, transparent groupings from ingredient text, designed for interpretability rather than perfect taxonomy.

In [96]:
ingredient_class_cols = [c for c in dataset.ingredient_group_features if c in recipe_summary.columns]

if ingredient_class_cols:
    class_summary = []
    for col in ingredient_class_cols:
        if col.endswith("_count"):
            class_summary.append({
                "feature": col,
                "component": feature_component_map.get(col, "ingredient_classes"),
                "mean": recipe_summary[col].mean(),
                "share_nonzero": (recipe_summary[col] > 0).mean(),
                "max": recipe_summary[col].max()
            })
    class_summary_df = pd.DataFrame(class_summary).sort_values("share_nonzero", ascending=False)
    display(class_summary_df)

    white_plot(
        px.bar(
            class_summary_df.sort_values("share_nonzero"),
            x="share_nonzero",
            y="feature",
            orientation="h",
            title="Ingredient-Class Presence Rate by Recipe"
        )
    )
else:
    print("No ingredient-class features were created. Check CONFIG.use_ingredient_group_features.")


,feature,component,mean,share_nonzero,max
15,ingredient_class_count,ingredient_classes,6.334316,0.999553,14.0
14,inggrp_beverages_alcohol_broth_count,ingredient_classes,2.358878,0.817435,14.0
9,inggrp_herbs_spices_seasonings_count,ingredient_classes,2.003573,0.809218,11.0
4,inggrp_vegetables_count,ingredient_classes,2.128796,0.654966,13.0
2,inggrp_dairy_count,ingredient_classes,1.179528,0.643444,9.0
6,inggrp_grains_starches_count,ingredient_classes,0.614952,0.520275,5.0
11,inggrp_sweeteners_chocolate_count,ingredient_classes,0.664523,0.444623,6.0
10,inggrp_oils_fats_count,ingredient_classes,0.482226,0.429171,4.0
0,inggrp_meat_poultry_count,ingredient_classes,0.534477,0.387549,6.0
3,inggrp_eggs_count,ingredient_classes,0.346820,0.328064,3.0


## Target distribution

In [97]:
white_plot(
    px.histogram(
        recipe_summary,
        x=CONFIG.target,
        nbins=30,
        title="Distribution of Recipe Ratings After Filtering"
    )
)

print(recipe_summary[CONFIG.target].describe())


count    11196.000000
mean         4.589282
std          0.240877
min          4.000000
25%          4.400000
50%          4.600000
75%          4.800000
max          5.000000
Name: rating_value, dtype: float64


# 6 — Data controls

In [98]:
controls = FeatureControls(CONFIG)
recipe_summary, continuous_features, binary_features, control_report = controls.apply(
    recipe_summary,
    continuous_features,
    binary_features
)
features = continuous_features + binary_features
feature_component_map = {feature: component for feature, component in feature_component_map.items() if feature in features}

component_overview_after = (
    pd.DataFrame({"feature": features})
    .assign(component=lambda d: d["feature"].map(feature_component_map).fillna("other"))
    .groupby("component", as_index=False)
    .agg(feature_count=("feature", "count"))
    .sort_values("feature_count", ascending=False)
)

print("Features after controls:")
print("Continuous features:", len(continuous_features))
print("Binary features:", len(binary_features))
print("Total final features:", len(features))
print("\nFeature components after controls:")
display(component_overview_after)

print("\nMissingness diagnostics:")
display(control_report["missing_table"])

print("\nLow-variance diagnostics:")
display(control_report["variance_table"].head(20))

print("\nOutlier diagnostics:")
display(control_report["outlier_table"])

print("\nHigh-correlation pairs:")
display(control_report["high_corr_table"])

print("\nVIF before filtering:")
display(control_report["vif_before"])

print("\nVIF after filtering:")
display(control_report["vif_after"])

if CONFIG.use_outlier_control and CONFIG.outlier_action == "winsorize":
    print("Outliers will be winsorized inside the model pipeline using train-set thresholds only.")


Features after controls:
Continuous features: 46
Binary features: 90
Total final features: 136

Feature components after controls:


,component,feature_count
0,individual_ingredients,60
1,ingredient_classes,33
2,metadata_context,17
5,recipe_complexity,15
3,nutrition,9
4,popularity_signal,2



Missingness diagnostics:


,feature,missing_rate
0,log_cook_minutes,0.145677
1,cook_minutes,0.145677
2,prep_minutes,0.027867
3,log_prep_minutes,0.027867
4,fat,0.025277
...,...,...
139,inggrp_meat_poultry_present,0.000000
140,log_rating_count,0.000000
141,rating_count,0.000000
142,publication_month,0.000000



Low-variance diagnostics:


...


VIF before filtering:


,feature,VIF
36,calories,301.773313
38,fat,107.524939
37,carbs,65.555556
39,protein,29.078872
46,carbs_per_calorie,28.208613
45,fat_per_calorie,22.671619
18,ingredient_count,22.100778
43,log_protein,18.443134
42,log_fat,15.763573
44,protein_per_calorie,14.849755



VIF after filtering:


,feature,VIF
0,ingredient_class_count,9.898798
38,log_calories,9.814312
42,carbs_per_calorie,9.565669
39,log_carbs,8.908479
43,calories_per_ingredient,6.488261
41,fat_per_calorie,6.094831
29,ingredients_per_step,5.954347
37,protein,5.861317
32,description_words_per_step,5.824545
26,direction_step_count,5.631437


Outliers will be winsorized inside the model pipeline using train-set thresholds only.


In [99]:
white_plot(
    px.bar(
        component_overview_after.sort_values("feature_count"),
        x="feature_count",
        y="component",
        orientation="h",
        title="Number of Features by Component After Controls"
    )
)

white_plot(
    px.bar(
        control_report["missing_table"],
        x="feature",
        y="missing_rate",
        title="Missing Rate by Feature"
    )
)

if len(control_report["outlier_table"]):
    white_plot(
        px.bar(
            control_report["outlier_table"],
            x="feature",
            y="outlier_rate",
            title="IQR Outlier Rate by Continuous Feature"
        )
    )

if not control_report["corr_matrix"].empty:
    white_plot(
        px.imshow(
            control_report["corr_matrix"],
            text_auto=".2f",
            title="Correlation Matrix — Continuous Features and Target"
        )
    )


# 7 — Train/test split and preprocessing

In [100]:
trainer = ModelTrainer(CONFIG)
X_train, X_test, y_train, y_test, w_train, w_test = trainer.train_test_data(recipe_summary, features)
preprocess = trainer.make_preprocess(continuous_features, binary_features)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Target range after filtering:", y_train.min(), "to", y_train.max())
print("Selected champion model:", CONFIG.champion_model_name)
print("Random Forest, Ridge, and Logistic Regression rating-band models are included as challengers.")
print("Feature components available for interpretation:", sorted(set(feature_component_map.values())))


Train shape: (8956, 136)
Test shape: (2240, 136)
Target range after filtering: 4.0 to 5.0
Selected champion model: Champion — Gradient Boosting Regressor
Random Forest, Ridge, and Logistic Regression rating-band models are included as challengers.
Feature components available for interpretation: ['individual_ingredients', 'ingredient_classes', 'metadata_context', 'nutrition', 'popularity_signal', 'recipe_complexity']


## Feature significance

In [101]:
if CONFIG.run_feature_significance:
    sig_preprocess = clone(preprocess)
    X_train_prepared = sig_preprocess.fit_transform(X_train)
    feature_names = [clean_feature_name(f) for f in sig_preprocess.get_feature_names_out()]

    X_train_prepared = np.asarray(X_train_prepared, dtype=float)
    X_train_prepared = np.nan_to_num(X_train_prepared, nan=0.0, posinf=0.0, neginf=0.0)

    f_values, p_values = f_regression(X_train_prepared, y_train)

    significance_df = (
        pd.DataFrame({
            "feature": feature_names,
            "component": [feature_component_map.get(f, "other") for f in feature_names],
            "F_stat": f_values,
            "p_value": p_values,
            "significant_5pct": p_values < CONFIG.significance_level
        })
        .replace([np.inf, -np.inf], np.nan)
        .sort_values("p_value")
    )

    significance_df["minus_log10_p"] = -np.log10(significance_df["p_value"].clip(lower=1e-300))

    print("Top statistically significant features using univariate F-test:")
    display(significance_df.head(30))

    component_significance = (
        significance_df
        .groupby("component", as_index=False)
        .agg(
            features=("feature", "count"),
            significant_features=("significant_5pct", "sum"),
            max_minus_log10_p=("minus_log10_p", "max"),
            median_minus_log10_p=("minus_log10_p", "median")
        )
        .sort_values("max_minus_log10_p", ascending=False)
    )

    print("\nFeature significance summarized by component:")
    display(component_significance)

    white_plot(
        px.bar(
            significance_df.head(25).sort_values("minus_log10_p"),
            x="minus_log10_p",
            y="feature",
            color="component",
            orientation="h",
            title="Top Feature Significance — Univariate F-test"
        )
    )

    white_plot(
        px.bar(
            component_significance.sort_values("max_minus_log10_p"),
            x="max_minus_log10_p",
            y="component",
            orientation="h",
            title="Strongest Univariate Signal by Feature Component"
        )
    )

    top_idx = significance_df.head(min(CONFIG.ols_max_features, len(significance_df))).index.tolist()
    X_ols = X_train_prepared[:, top_idx]
    ols_feature_names = significance_df.loc[top_idx, "feature"].tolist()

    try:
        import statsmodels.api as sm

        X_ols_const = sm.add_constant(X_ols)
        ols_model = sm.OLS(y_train.values, X_ols_const).fit(cov_type="HC3")

        ols_df = pd.DataFrame({
            "feature": ["const"] + ols_feature_names,
            "component": ["intercept"] + [feature_component_map.get(f, "other") for f in ols_feature_names],
            "coef": ols_model.params,
            "p_value": ols_model.pvalues,
            "significant_5pct": ols_model.pvalues < CONFIG.significance_level
        }).sort_values("p_value")

        print("Multivariate OLS significance with robust HC3 standard errors:")
        display(ols_df)

        white_plot(
            px.bar(
                ols_df.query("feature != 'const'").head(25).sort_values("coef"),
                x="coef",
                y="feature",
                orientation="h",
                color="component",
                title="Multivariate OLS Coefficients by Component"
            )
        )

    except Exception as e:
        print("Statsmodels OLS significance skipped because statsmodels is unavailable or failed.")
        print("Reason:", e)
else:
    print("Feature significance skipped. Set CONFIG.run_feature_significance=True to run it.")


Top statistically significant features using univariate F-test:


,feature,component,F_stat,p_value,significant_5pct,minus_log10_p
26,direction_step_count,recipe_complexity,212.171881,1.608782e-47,True,46.793503
33,publication_year,metadata_context,174.940215,1.448054e-39,True,38.839215
32,description_words_per_step,recipe_complexity,80.909514,2.846413e-19,True,18.545702
1,ingredient_class_diversity,ingredient_classes,77.553838,1.532784e-18,True,17.814519
91,ing_kosher_salt,individual_ingredients,75.384818,4.554550e-18,True,17.341555
12,inggrp_herbs_spices_seasonings_count,ingredient_classes,67.699945,2.170289e-16,True,15.663482
30,steps_per_ingredient,recipe_complexity,60.337703,8.870624e-15,True,14.052046
8,inggrp_fruit_count,ingredient_classes,59.821843,1.150853e-14,True,13.938980
111,ing_salt_and_pepper,individual_ingredients,59.223660,1.556524e-14,True,13.807844
27,category_count,recipe_complexity,47.910153,4.768886e-12,True,11.321583



Feature significance summarized by component:


,component,features,significant_features,max_minus_log10_p,median_minus_log10_p
5,recipe_complexity,15,11,46.793503,5.898576
2,metadata_context,17,5,38.839215,0.851564
1,ingredient_classes,33,17,17.814519,1.558390
0,individual_ingredients,60,26,17.341555,1.138408
3,nutrition,9,6,6.500565,2.479241
4,popularity_signal,2,2,2.537189,2.289309


Multivariate OLS significance with robust HC3 standard errors:


,feature,component,coef,p_value,significant_5pct
0,const,intercept,4.607728,0.000000e+00,True
2,publication_year,metadata_context,0.021518,2.037133e-20,True
9,ing_salt_and_pepper,individual_ingredients,-0.070580,1.034181e-10,True
25,protein_per_calorie,nutrition,-0.024395,1.213159e-09,True
21,fat_per_calorie,nutrition,0.016069,4.022100e-05,True
1,direction_step_count,recipe_complexity,0.034182,1.001835e-04,True
7,steps_per_ingredient,recipe_complexity,0.029571,1.049137e-04,True
14,ing_egg,individual_ingredients,-0.026588,1.809567e-04,True
20,ing_onion,individual_ingredients,-0.027280,3.312979e-04,True
16,ingredients_per_step,recipe_complexity,0.028168,5.345531e-04,True


# 8 — Champion/challenger model comparison

In [102]:
results_df, fitted_models, champion_model, champion_row = trainer.fit_evaluate(
    preprocess,
    X_train,
    X_test,
    y_train,
    y_test,
    w_train=w_train
)

print("Model comparison, sorted by test R²:")
display(results_df)

print(f"Selected champion model: {CONFIG.champion_model_name}")
print(f"Champion test R²: {champion_row['test_R2']:.4f}")
print(f"Champion test MAE: {champion_row['test_MAE']:.4f}")

white_plot(
    px.bar(
        results_df.sort_values("test_R2"),
        x="test_R2",
        y="model",
        orientation="h",
        title="Model Comparison — Test R² Higher is Better"
    )
)

white_plot(
    px.bar(
        results_df.sort_values("test_MAE", ascending=False),
        x="test_MAE",
        y="model",
        orientation="h",
        title="Model Comparison — Test MAE Lower is Better"
    )
)


Training Baseline — Mean...
Training Champion — Gradient Boosting Regressor...
Training Challenger 1 — Ridge...
Training Challenger 2 — Logistic Regression...
Training Challenger 3 — Random Forest Regressor...
Model comparison, sorted by test R²:


,model,cv_MAE,cv_RMSE,cv_R2,train_MAE,train_RMSE,train_R2,train_Within_0.10,train_Within_0.25,train_Within_0.50,test_MAE,test_RMSE,test_R2,test_Within_0.10,test_Within_0.25,test_Within_0.50
1,Champion — Gradient Boosting Regressor,0.176984,0.225064,0.125488,0.169447,0.216507,0.190772,0.376172,0.757034,0.972867,0.177759,0.226468,0.121407,0.370536,0.744196,0.967857
4,Challenger 3 — Random Forest Regressor,0.178539,0.225942,0.118681,0.110442,0.144448,0.639794,0.570679,0.909893,0.999218,0.179190,0.227368,0.114409,0.355804,0.736607,0.970982
2,Challenger 1 — Ridge,0.180775,0.229365,0.091788,0.177551,0.225964,0.118535,0.360652,0.740063,0.969518,0.180468,0.230492,0.089912,0.363393,0.737946,0.963839
3,Challenger 2 — Logistic Regression,0.180902,0.230430,0.083337,0.177702,0.227993,0.102639,0.366682,0.741402,0.963153,0.181426,0.232812,0.071498,0.363393,0.739286,0.958482
0,Baseline — Mean,0.190698,0.240674,-0.000014,0.191370,0.240695,-0.000135,0.294105,0.715610,0.968513,0.192515,0.241610,-0.000011,0.294196,0.713393,0.969196


Selected champion model: Champion — Gradient Boosting Regressor
Champion test R²: 0.1214
Champion test MAE: 0.1778


# 9 — Champion model diagnostics

In [103]:
champion_pred = np.clip(
    champion_model.predict(X_test),
    CONFIG.min_target_value if CONFIG.filter_low_ratings else 1.0,
    CONFIG.max_target_value
)

error_df = pd.DataFrame({
    "actual": y_test,
    "predicted": champion_pred,
    "residual": y_test - champion_pred,
    "absolute_error": abs(y_test - champion_pred)
})

error_df["actual_rating_band"] = pd.cut(
    error_df["actual"],
    bins=[CONFIG.min_target_value, 4.25, 4.50, 4.75, CONFIG.max_target_value],
    labels=["4.00-4.25", "4.25-4.50", "4.50-4.75", "4.75-5.00"],
    include_lowest=True
)

print("Selected champion model:", CONFIG.champion_model_name)
display(error_df.describe())

white_plot(px.scatter(error_df, x="actual", y="predicted", trendline="ols", title=f"Actual vs Predicted Ratings — {CONFIG.champion_model_name}"))
white_plot(px.histogram(error_df, x="residual", nbins=40, title=f"Residual Distribution — {CONFIG.champion_model_name}"))
white_plot(px.box(error_df, x="actual_rating_band", y="absolute_error", title=f"Absolute Error by Rating Band — {CONFIG.champion_model_name}"))


Selected champion model: Champion — Gradient Boosting Regressor


,actual,predicted,residual,absolute_error
count,2240.000000,2240.000000,2240.000000,2240.000000
mean,4.587679,4.590044,-0.002365,0.177759
std,0.241663,0.074781,0.226506,0.140351
min,4.000000,4.320293,-0.786696,0.000115
25%,4.400000,4.543493,-0.130947,0.068429
50%,4.600000,4.595033,0.028710,0.145360
75%,4.800000,4.638018,0.156292,0.253752
max,5.000000,4.803529,0.637778,0.786696


## Champion feature importance

In [104]:
try:
    fitted_preprocess = champion_model.named_steps["preprocess"]
    feature_names = [clean_feature_name(f) for f in fitted_preprocess.get_feature_names_out()]
    fitted_estimator = champion_model.named_steps["model"]

    if hasattr(fitted_estimator, "feature_importances_"):
        importance_df = (
            pd.DataFrame({"feature": feature_names, "importance": fitted_estimator.feature_importances_})
            .assign(component=lambda d: d["feature"].map(feature_component_map).fillna("other"))
            .sort_values("importance", ascending=False)
        )

        component_importance = (
            importance_df
            .groupby("component", as_index=False)
            .agg(
                total_importance=("importance", "sum"),
                avg_importance=("importance", "mean"),
                top_feature=("feature", "first"),
                feature_count=("feature", "count")
            )
            .sort_values("total_importance", ascending=False)
        )

        print("Top champion model feature importances:")
        display(importance_df.head(30))

        print("\nChampion feature importance grouped by component:")
        display(component_importance)

        white_plot(
            px.bar(
                importance_df.head(25).sort_values("importance"),
                x="importance",
                y="feature",
                color="component",
                orientation="h",
                title=f"Top Feature Importances — {CONFIG.champion_model_name}"
            )
        )

        white_plot(
            px.bar(
                component_importance.sort_values("total_importance"),
                x="total_importance",
                y="component",
                orientation="h",
                title=f"Total Feature Importance by Component — {CONFIG.champion_model_name}"
            )
        )

    elif hasattr(fitted_estimator, "coef_"):
        coef_df = (
            pd.DataFrame({"feature": feature_names, "coefficient": np.ravel(fitted_estimator.coef_)})
            .assign(
                abs_coefficient=lambda d: d["coefficient"].abs(),
                component=lambda d: d["feature"].map(feature_component_map).fillna("other")
            )
            .sort_values("abs_coefficient", ascending=False)
        )

        print("Top champion model coefficients:")
        display(coef_df.head(30))
    else:
        print("The champion model does not expose feature_importances_ or coef_.")

except Exception as e:
    print("Feature importance extraction skipped.")
    print("Reason:", e)


Top champion model feature importances:


,feature,importance,component
33,publication_year,0.152423,metadata_context
26,direction_step_count,0.120267,recipe_complexity
24,description_word_count,0.095371,recipe_complexity
45,log_rating_count,0.087316,popularity_signal
40,protein_per_calorie,0.071519,nutrition
41,fat_per_calorie,0.045972,nutrition
44,rating_count,0.044111,popularity_signal
1,ingredient_class_diversity,0.027665,ingredient_classes
31,total_minutes_per_ingredient,0.025876,recipe_complexity
52,inggrp_grains_starches_present,0.022474,ingredient_classes



Champion feature importance grouped by component:


,component,total_importance,avg_importance,top_feature,feature_count
5,recipe_complexity,0.309629,0.020642,direction_step_count,15
3,nutrition,0.195510,0.021723,protein_per_calorie,9
2,metadata_context,0.160483,0.009440,publication_year,17
4,popularity_signal,0.131427,0.065714,log_rating_count,2
1,ingredient_classes,0.129609,0.003928,ingredient_class_diversity,33
0,individual_ingredients,0.073342,0.001222,ing_salt_and_pepper,60


# 10 — Descriptive analysis by ingredient class

In [105]:
low_rating_threshold = 4.25
min_recipe_count = 30

# Ingredient-class descriptive analysis at recipe level
class_count_cols = [c for c in dataset.ingredient_group_features if c.endswith("_count") and c in recipe_summary.columns]

class_rating_rows = []
for col in class_count_cols:
    recipes_with_class = recipe_summary[recipe_summary[col] > 0]
    if len(recipes_with_class) >= min_recipe_count:
        class_rating_rows.append({
            "ingredient_class": col.replace("inggrp_", "").replace("_count", ""),
            "recipe_count": len(recipes_with_class),
            "avg_rating": recipes_with_class[CONFIG.target].mean(),
            "median_rating": recipes_with_class[CONFIG.target].median(),
            "lower_high_range_share": (recipes_with_class[CONFIG.target] < low_rating_threshold).mean(),
            "avg_ingredient_count": recipes_with_class["ingredient_count"].mean()
        })

class_rating_df = pd.DataFrame(class_rating_rows).sort_values("avg_rating")
print("Ingredient-class rating summary:")
display(class_rating_df)

if len(class_rating_df):
    white_plot(
        px.bar(
            class_rating_df.sort_values("avg_rating"),
            x="avg_rating",
            y="ingredient_class",
            orientation="h",
            hover_data=["recipe_count", "lower_high_range_share", "avg_ingredient_count"],
            title="Average Rating by Ingredient Class"
        )
    )

# Keep individual-ingredient analysis as a detailed appendix.
ingredient_rating = ingredients_df.copy()
ingredient_rating["ingredient_canonical"] = (
    ingredient_rating["ingredient_canonical"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)

ingredient_rating = ingredient_rating[ingredient_rating["rating_value"] >= CONFIG.min_target_value].copy()
ingredient_rating["is_lower_within_high_range"] = ingredient_rating["rating_value"] < low_rating_threshold

ingredient_summary = (
    ingredient_rating
    .groupby("ingredient_canonical", as_index=False)
    .agg(
        recipe_count=("recipe_id", "nunique"),
        avg_rating=("rating_value", "mean"),
        lower_high_range_recipes=("is_lower_within_high_range", "sum")
    )
)

ingredient_summary["lower_high_range_share"] = ingredient_summary["lower_high_range_recipes"] / ingredient_summary["recipe_count"]

ingredient_summary = (
    ingredient_summary[
        (ingredient_summary["ingredient_canonical"] != "")
        & (ingredient_summary["recipe_count"] >= min_recipe_count)
    ]
    .sort_values("avg_rating")
)

print("\nLowest-rated individual ingredients within the 4+ modeling range:")
display(ingredient_summary.head(15))


Ingredient-class rating summary:


,ingredient_class,recipe_count,avg_rating,median_rating,lower_high_range_share,avg_ingredient_count
7,legumes_soy,1938,4.569556,4.6,0.114035,11.031476
6,grains_starches,5825,4.574541,4.6,0.106781,10.526524
3,eggs,3673,4.577348,4.6,0.108631,10.355568
1,seafood,732,4.580738,4.6,0.101093,11.214481
2,dairy,7204,4.585966,4.6,0.102860,9.779845
13,baking_additives,3058,4.588031,4.6,0.105625,10.114454
4,vegetables,7333,4.588354,4.6,0.096277,10.420974
15,ingredient_class,11191,4.589295,4.6,0.098293,9.569922
0,meat_poultry,4339,4.589998,4.6,0.091265,10.821387
14,beverages_alcohol_broth,9152,4.592177,4.6,0.098011,10.241587



Lowest-rated individual ingredients within the 4+ modeling range:


,ingredient_canonical,recipe_count,avg_rating,lower_high_range_recipes,lower_high_range_share
991,10.75 ounce can condensed cream of mushroom soup,75,4.397333,20,0.266667
990,10.75 ounce can condensed cream of chicken soup,39,4.435897,6,0.153846
6611,bread crumb,63,4.469841,14,0.222222
636,10 inch flour tortilla,49,4.483673,4,0.081633
19957,uncooked white rice,62,4.496774,8,0.129032
9389,dry red wine,34,4.500000,7,0.205882
5541,applesauce,32,4.500000,8,0.250000
6679,broccoli floret,37,4.502703,6,0.162162
16156,potatoes peeled and cubed,40,4.505000,7,0.175000
6268,black olive,41,4.507317,8,0.195122


# 11 — Export model files for Streamlit demo

This section saves the fitted **Gradient Boosting champion pipeline** and the feature information needed by a Streamlit app. The app can then load the trained model, accept recipe ingredients/nutrition inputs, predict a rating, and use actual SHAP values to explain which features push the prediction up or down.

Run this cell after the champion model has been trained.

In [106]:
import json
import joblib
from pathlib import Path

STREAMLIT_DIR = Path("streamlit_artifacts")
STREAMLIT_DIR.mkdir(exist_ok=True)

# Save the full fitted pipeline, including preprocessing + Gradient Boosting model.
joblib.dump(champion_model, STREAMLIT_DIR / "rating_gradient_boosting_pipeline.pkl")

# Save the raw input feature columns expected before preprocessing.
joblib.dump(features, STREAMLIT_DIR / "rating_model_raw_features.pkl")

# Save the processed feature names after preprocessing. These are used for SHAP display.
fitted_preprocess = champion_model.named_steps["preprocess"]
processed_feature_names = [clean_feature_name(f) for f in fitted_preprocess.get_feature_names_out()]
joblib.dump(processed_feature_names, STREAMLIT_DIR / "rating_model_processed_features.pkl")

# Save a small raw training sample for SHAP background data.
shap_background_raw = X_train.sample(min(200, len(X_train)), random_state=CONFIG.random_state)
shap_background_raw.to_csv(STREAMLIT_DIR / "shap_background_raw.csv", index=False)

# Save feature-component labels for optional interpretation in the app.
feature_component_df = (
    pd.DataFrame({"feature": features})
    .assign(component=lambda d: d["feature"].map(feature_component_map).fillna("other"))
)
feature_component_df.to_csv(STREAMLIT_DIR / "feature_component_map.csv", index=False)

# Save model metadata for the app/report.
metadata = {
    "champion_model_name": CONFIG.champion_model_name,
    "target": CONFIG.target,
    "min_target_value": CONFIG.min_target_value if CONFIG.filter_low_ratings else 1.0,
    "max_target_value": CONFIG.max_target_value,
    "test_MAE": float(champion_row["test_MAE"]),
    "test_RMSE": float(champion_row["test_RMSE"]),
    "test_R2": float(champion_row["test_R2"]),
    "test_Within_0.10": float(champion_row["test_Within_0.10"]),
    "test_Within_0.25": float(champion_row["test_Within_0.25"]),
    "test_Within_0.50": float(champion_row["test_Within_0.50"]),
}
with open(STREAMLIT_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved Streamlit files to:", STREAMLIT_DIR.resolve())
print("Files created:")
for path in sorted(STREAMLIT_DIR.iterdir()):
    print("-", path.name)


Saved Streamlit files to: /Users/evelyn/Desktop/M2/machine_learning/src/streamlit_artifacts
Files created:
- feature_component_map.csv
- model_metadata.json
- rating_gradient_boosting_pipeline.pkl
- rating_model_processed_features.pkl
- rating_model_raw_features.pkl
- shap_background_raw.csv


# 15 — Streamlit app
This cell writes a file called `streamlit_app.py`. Put it in the same folder as the `streamlit_artifacts` folder, then run:

```bash
streamlit run streamlit_app.py
```

Install packages first if needed:

```bash
python -m pip install streamlit pandas numpy scikit-learn joblib shap matplotlib
```

In [108]:
from pathlib import Path

streamlit_app_code = r'''
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import streamlit as st
from sklearn.base import BaseEstimator, TransformerMixin


ARTIFACT_DIR = Path("streamlit_artifacts")

st.set_page_config(
    page_title="Recipe Rating Predictor",
    page_icon="🍽️",
    layout="wide"
)

st.markdown("""
<style>
.block-container {
    padding-top: 1.5rem;
    max-width: 1100px;
}

[data-testid="stSidebar"] {
    background-color: #fff7ed;
}

.hero {
    background: white;
    padding: 1.4rem 1.6rem;
    border-radius: 18px;
    box-shadow: 0 4px 18px rgba(0,0,0,0.07);
    margin-bottom: 1.2rem;
}

.hero h1 {
    margin-bottom: 0.3rem;
    font-size: 2.4rem;
}

.hero p {
    color: #666;
    font-size: 1rem;
}

.rating-card {
    background: #fff7ed;
    border: 1px solid #fed7aa;
    padding: 1.4rem;
    border-radius: 18px;
    text-align: center;
    margin-bottom: 1.2rem;
}

.rating-number {
    font-size: 3.2rem;
    font-weight: 800;
    color: #c2410c;
}

.driver-card {
    background: white;
    padding: 1rem;
    border-radius: 14px;
    border: 1px solid #eee;
    margin-bottom: 0.7rem;
}
</style>
""", unsafe_allow_html=True)


class IQRClipper(BaseEstimator, TransformerMixin):
    def __init__(self, k=1.5, enabled=True):
        self.k = k
        self.enabled = enabled

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        q1 = np.nanpercentile(X, 25, axis=0)
        q3 = np.nanpercentile(X, 75, axis=0)
        iqr = q3 - q1
        self.low_ = q1 - self.k * iqr
        self.high_ = q3 + self.k * iqr
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        if not self.enabled:
            return X
        return np.clip(X, self.low_, self.high_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)


@st.cache_resource
def load_artifacts():
    model_pipeline = joblib.load(ARTIFACT_DIR / "rating_gradient_boosting_pipeline.pkl")
    raw_features = joblib.load(ARTIFACT_DIR / "rating_model_raw_features.pkl")
    processed_features = joblib.load(ARTIFACT_DIR / "rating_model_processed_features.pkl")
    background_raw = pd.read_csv(ARTIFACT_DIR / "shap_background_raw.csv")

    with open(ARTIFACT_DIR / "model_metadata.json", "r") as f:
        metadata = json.load(f)

    return model_pipeline, raw_features, processed_features, background_raw, metadata


model_pipeline, raw_features, processed_features, background_raw, metadata = load_artifacts()

preprocess = model_pipeline.named_steps["preprocess"]
estimator = model_pipeline.named_steps["model"]


def make_safe_column_name(text, prefix="ing_"):
    text = str(text).lower().strip()
    text = "".join(ch if ch.isalnum() else "_" for ch in text)
    text = "_".join(part for part in text.split("_") if part)
    return prefix + text


def available_ingredient_names():
    names = []
    for col in raw_features:
        if col.startswith("ing_") and not col.startswith("inggrp_"):
            names.append(col.replace("ing_", "").replace("_", " "))
    return sorted(names)


def build_input_row(typed_ingredients, clicked_ingredients, calories, carbs, fat, protein):
    row = pd.DataFrame(0.0, index=[0], columns=raw_features)

    ingredients = []
    ingredients.extend([x.strip().lower() for x in typed_ingredients.split(",") if x.strip()])
    ingredients.extend([x.strip().lower() for x in clicked_ingredients])
    ingredients = sorted(set(ingredients))

    ingredient_count = len(ingredients)

    values = {
        "calories": calories,
        "carbs": carbs,
        "fat": fat,
        "protein": protein,
        "log_calories": np.log1p(max(calories, 0)),
        "log_carbs": np.log1p(max(carbs, 0)),
        "log_fat": np.log1p(max(fat, 0)),
        "log_protein": np.log1p(max(protein, 0)),
        "ingredient_count": ingredient_count,
        "log_ingredient_count": np.log1p(max(ingredient_count, 0)),
        "protein_per_calorie": protein / calories if calories else 0,
        "fat_per_calorie": fat / calories if calories else 0,
        "carbs_per_calorie": carbs / calories if calories else 0,
        "calories_per_ingredient": calories / ingredient_count if ingredient_count else 0,
    }

    for col, value in values.items():
        if col in row.columns:
            row.loc[0, col] = value

    for ingredient in ingredients:
        safe_col = make_safe_column_name(ingredient, prefix="ing_")
        if safe_col in row.columns:
            row.loc[0, safe_col] = 1.0

    return row, ingredients


def shap_explain(input_row):
    background_raw_aligned = background_raw.reindex(columns=raw_features, fill_value=0)
    input_row_aligned = input_row.reindex(columns=raw_features, fill_value=0)

    background_processed = preprocess.transform(background_raw_aligned)
    input_processed = preprocess.transform(input_row_aligned)

    explainer = shap.TreeExplainer(estimator, background_processed)
    shap_values = explainer.shap_values(input_processed)

    expected_value = float(np.ravel(explainer.expected_value)[0])
    shap_values_1d = np.ravel(shap_values)
    input_values_1d = np.ravel(input_processed)

    feature_clean_names = (
        pd.Series(processed_features)
        .str.replace("ing_", "", regex=False)
        .str.replace("inggrp_", "ingredient group: ", regex=False)
        .str.replace("_", " ", regex=False)
        .tolist()
    )

    shap_df = pd.DataFrame({
        "feature": processed_features,
        "feature_clean": feature_clean_names,
        "value": input_values_1d,
        "shap_value": shap_values_1d,
    })

    shap_df = shap_df.reindex(
        shap_df["shap_value"].abs().sort_values(ascending=False).index
    )

    shap_explanation = shap.Explanation(
        values=shap_values_1d,
        base_values=expected_value,
        data=input_values_1d,
        feature_names=feature_clean_names
    )

    return shap_df, expected_value, shap_explanation


with st.sidebar:
    st.header("Recipe input")

    typed_ingredients = st.text_area(
        "Type ingredients separated by commas",
        value="chicken, garlic, onion, olive oil, tomato, salt, pepper"
    )

    clicked_ingredients = st.multiselect(
        "Click extra known ingredients",
        options=available_ingredient_names(),
        default=[]
    )

    st.markdown("---")
    st.subheader("Optional nutrition")

    calories = st.number_input("Calories", min_value=0.0, value=400.0)
    carbs = st.number_input("Carbs (g)", min_value=0.0, value=30.0)
    fat = st.number_input("Fat (g)", min_value=0.0, value=15.0)
    protein = st.number_input("Protein (g)", min_value=0.0, value=25.0)

    predict_clicked = st.button("Predict rating", use_container_width=True)


st.markdown("""
<div class="hero">
    <h1>🍽️ Recipe Rating Predictor</h1>
    <p>Enter ingredients to predict a recipe rating and see the main SHAP drivers.</p>
</div>
""", unsafe_allow_html=True)

st.caption(
    f"Model: {metadata['champion_model_name']} | "
    f"MAE: {metadata['test_MAE']:.3f} | "
    f"RMSE: {metadata['test_RMSE']:.3f} | "
    f"R²: {metadata['test_R2']:.3f}"
)

if predict_clicked:
    input_row, ingredients = build_input_row(
        typed_ingredients=typed_ingredients,
        clicked_ingredients=clicked_ingredients,
        calories=calories,
        carbs=carbs,
        fat=fat,
        protein=protein,
    )

    prediction = float(model_pipeline.predict(input_row)[0])

    if "min_target_value" in metadata and "max_target_value" in metadata:
        prediction = min(max(prediction, metadata["min_target_value"]), metadata["max_target_value"])

    shap_df, expected_value, shap_explanation = shap_explain(input_row)

    st.markdown(f"""
    <div class="rating-card">
        <div>Predicted recipe rating</div>
        <div class="rating-number">{prediction:.2f} / 5</div>
    </div>
    """, unsafe_allow_html=True)

    st.write("**Ingredients used:** " + (", ".join(ingredients) if ingredients else "None"))

    st.subheader("What is driving this rating?")
    
    # Only show ingredients that were actually entered/selected
    ingredient_shap_df = shap_df[
        (shap_df["feature"].str.startswith("ing_")) &
        (shap_df["value"] > 0)
    ].copy()
    
    ingredient_plot_df = ingredient_shap_df.sort_values("shap_value")
    
    if ingredient_plot_df.empty:
        st.warning("None of the entered ingredients matched the model's known ingredient features.")
    else:
        fig, ax = plt.subplots(figsize=(8, 4.5))
        ax.barh(ingredient_plot_df["feature_clean"], ingredient_plot_df["shap_value"])
        ax.axvline(0)
        ax.set_xlabel("SHAP value")
        ax.set_ylabel("Ingredient")
        ax.set_title("Ingredients driving the predicted rating")
        st.pyplot(fig)
        plt.close(fig)
    
    positive = ingredient_shap_df[
        ingredient_shap_df["shap_value"] > 0
    ].sort_values("shap_value", ascending=False).head(5)
    
    negative = ingredient_shap_df[
        ingredient_shap_df["shap_value"] < 0
    ].sort_values("shap_value").head(5)

    col1, col2 = st.columns(2)

    with col1:
        st.markdown("#### Pushing rating up")
        for _, row in positive.iterrows():
            st.markdown(
                f"<div class='driver-card'>⬆️ <b>{row['feature_clean']}</b><br>"
                f"SHAP: {row['shap_value']:.3f}</div>",
                unsafe_allow_html=True
            )

    with col2:
        st.markdown("#### Pushing rating down")
        for _, row in negative.iterrows():
            st.markdown(
                f"<div class='driver-card'>⬇️ <b>{row['feature_clean']}</b><br>"
                f"SHAP: {row['shap_value']:.3f}</div>",
                unsafe_allow_html=True
            )

else:
    st.info("Enter ingredients in the sidebar and click Predict rating.")
'''

Path("streamlit_app.py").write_text(streamlit_app_code)

print("Created simplified streamlit_app.py")
print("Run with: streamlit run streamlit_app.py")

Created simplified streamlit_app.py
Run with: streamlit run streamlit_app.py
